# Challenge 3 — Developing Multi-Agent Systems

This notebook extends the weather agent with a Google Search sub-agent and a coordinating root agent. The Google Maps key is requested only at runtime and is never saved in the notebook.

In [ ]:
%pip install -q --upgrade google-adk requests

import os
from getpass import getpass
from typing import Dict, List, Optional

import requests
import vertexai
from google.adk.agents import LlmAgent
from google.adk.tools import agent_tool, google_search
from vertexai.preview import reasoning_engines

PROJECT_ID = "qwiklabs-gcp-02-9e12deb8c42f"
LOCATION = "us-central1"
MODEL_GEMINI = "gemini-2.5-flash"

vertexai.init(project=PROJECT_ID, location=LOCATION)
print("Setup complete.")

In [ ]:
GOOGLE_MAPS_API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY")
if not GOOGLE_MAPS_API_KEY:
    GOOGLE_MAPS_API_KEY = getpass("Paste your Google Maps API key: ")
    os.environ["GOOGLE_MAPS_API_KEY"] = GOOGLE_MAPS_API_KEY

NWS_HEADERS = {"User-Agent": "ReadyNowWeatherAgent/1.0 (student lab)"}


def get_lat_lon(location: str) -> Optional[Dict[str, float]]:
    """Convert a United States location into latitude and longitude."""
    response = requests.get(
        "https://maps.googleapis.com/maps/api/geocode/json",
        params={"address": location, "components": "country:US", "key": GOOGLE_MAPS_API_KEY},
        timeout=20,
    )
    response.raise_for_status()
    payload = response.json()
    if payload.get("status") != "OK" or not payload.get("results"):
        return None
    result = payload["results"][0]
    coordinates = result["geometry"]["location"]
    return {
        "latitude": float(coordinates["lat"]),
        "longitude": float(coordinates["lng"]),
        "formatted_address": result["formatted_address"],
    }


def get_extended_weather_forecast(lat: float, lon: float) -> List[Dict[str, str]]:
    """Return up to six National Weather Service forecast periods for U.S. coordinates."""
    point_response = requests.get(
        f"https://api.weather.gov/points/{lat},{lon}", headers=NWS_HEADERS, timeout=20
    )
    point_response.raise_for_status()
    forecast_url = point_response.json()["properties"]["forecast"]
    forecast_response = requests.get(forecast_url, headers=NWS_HEADERS, timeout=20)
    forecast_response.raise_for_status()
    return [
        {
            "period": period["name"],
            "temperature": f"{period['temperature']} {period['temperatureUnit']}",
            "wind": f"{period['windSpeed']} {period['windDirection']}",
            "forecast": period["shortForecast"],
            "detail": period["detailedForecast"],
        }
        for period in forecast_response.json()["properties"]["periods"][:6]
    ]

print("Weather tools created.")

In [ ]:
WEATHER_AGENT_INSTRUCTIONS = """You are Pat, a careful U.S. real-time weather-alert agent.
Use get_lat_lon for a U.S. location, then get_extended_weather_forecast for its coordinates.
Summarize the near-term forecast clearly and highlight hazards. Do not invent weather information.
NWS only covers the United States, so ask for a U.S. city/state when needed."""

weather_agent = LlmAgent(
    name="weather_agent",
    model=MODEL_GEMINI,
    description="Provides real-time National Weather Service forecasts and weather alerts for U.S. locations.",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_lat_lon, get_extended_weather_forecast],
)

search_agent = LlmAgent(
    name="search_agent",
    model=MODEL_GEMINI,
    description="Finds current, reliable web information using Google Search.",
    instruction=(
        "Use Google Search for current general-information requests that are not U.S. weather forecasts. "
        "Give a concise, source-grounded answer and do not invent search results."
    ),
    tools=[google_search],
)

root_agent = LlmAgent(
    name="root_agent",
    model=MODEL_GEMINI,
    description="Coordinates weather and search specialists.",
    instruction=(
        "You are the coordinating agent. Delegate U.S. weather forecasts and weather alerts to weather_agent. "
        "Delegate current general-information requests to search_agent. "
        "Use one specialist at a time unless the user clearly needs both, then combine the results clearly."
    ),
    # Google Search is wrapped as an agent tool because this runtime does not
    # support Google Search and automatic sub-agent routing in one model call.
    tools=[agent_tool.AgentTool(agent=search_agent)],
    sub_agents=[weather_agent],
)

print("Created weather_agent, search_agent, and root_agent.")

In [ ]:
# Tool-level smoke test for the weather specialist.
for location in ["New York, NY", "Miami, FL", "Denver, CO"]:
    coordinates = get_lat_lon(location)
    assert coordinates is not None, f"No coordinates found for {location}"
    forecast = get_extended_weather_forecast(coordinates["latitude"], coordinates["longitude"])
    assert forecast, f"No forecast found for {location}"
    print(f"{coordinates['formatted_address']}: {forecast[0]['period']} — {forecast[0]['forecast']}")

In [ ]:
# Root-agent tests. Event output demonstrates delegation to the appropriate sub-agent.
multi_agent_app = reasoning_engines.AdkApp(agent=root_agent)

TEST_REQUESTS = [
    "Give me a concise weather alert for Miami, Florida.",
    "What are the latest safety recommendations for preparing a hurricane kit?",
]

for index, request in enumerate(TEST_REQUESTS, start=1):
    session = multi_agent_app.create_session(user_id="challenge-three-tester")
    session_id = session["id"] if isinstance(session, dict) else session.id
    print(f"\n=== Test {index}: {request} ===")
    for event in multi_agent_app.stream_query(
        user_id="challenge-three-tester",
        session_id=session_id,
        message=request,
    ):
        author = event.get("author", "unknown") if isinstance(event, dict) else getattr(event, "author", "unknown")
        content = event.get("content") if isinstance(event, dict) else getattr(event, "content", None)
        parts = content.get("parts", []) if isinstance(content, dict) else getattr(content, "parts", [])
        text = " ".join(
            part.get("text", "") if isinstance(part, dict) else getattr(part, "text", "")
            for part in parts
        ).strip()
        if text:
            print(f"[{author}] {text}")

## Agent architecture

```
User request
     ↓
root_agent (coordinator)
   ├─ weather_agent (sub-agent) → Google Maps Geocoding → National Weather Service
   └─ search_agent (agent tool) → Google Search
     ↓
Combined, concise response
```

The root agent delegates weather requests to its weather sub-agent. It invokes the Google Search specialist through an ADK `AgentTool`, which is the compatible pattern for this runtime. The test cell prints event authors for both routes.